# Bibliotecas e parâmetros

## Importação de bibliotecas

In [ ]:
import math
import pathlib
from datetime import date

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from folium.plugins import HeatMap
from matplotlib import dates as mdates

## Parâmetros

In [ ]:
# Parâmetros de extração
DIR_DADOS_UNIFICADOS: str = "../data/interim"
DIR_DADOS_PROCESSADOS: str = "../data/processed"
NOME_ARQUIVO_DATASET_CONSOLIDADO: str = "SPDadosCriminais_Unificado.csv"
NOME_ARQUIVO_DATASET_PROCESSADO: str = "dados_processados.csv"
COLUNAS_DESEJADAS: list[str] = [
    "NUM_BO",
    "BAIRRO",
    "LATITUDE",
    "LONGITUDE",
    "DATA_COMUNICACAO_BO",
    "DATA_OCORRENCIA_BO",
    "HORA_OCORRENCIA_BO",
    "DESCR_PERIODO",
    "RUBRICA",
    "ano_referencia",
]

# Parâmetros de filtragem do universo
COORDENADAS_ORIGEM: tuple[float, float] = (-23.5237759, -46.624897)
RAIO_ANALISE_KM: float = 2.0

# Extração dos dados

## Carregando dataset em um DataFrame do Pandas

In [ ]:
df_crimes = pd.read_csv(
    f"{DIR_DADOS_UNIFICADOS}/{NOME_ARQUIVO_DATASET_CONSOLIDADO}",
    sep=";",
    usecols=COLUNAS_DESEJADAS,
    low_memory=False,
)

In [ ]:
df_crimes.head()

## Filtrando universo

In [ ]:
def filtrar_por_raio(
    df: pd.DataFrame,
    lat_col: str,
    lon_col: str,
    centro_lat: float,
    centro_lon: float,
    raio_km: float = 2.0,
    incluir_distancia: bool = True,
):
    """Filtra linhas de um DataFrame dentro de um raio (em km) a partir de um ponto central."""
    # Raio médio da Terra em quilômetros
    raio_terra = 6371.0088

    # Conversão de graus para radianos
    lat1 = np.radians(centro_lat)
    lon1 = np.radians(centro_lon)
    lat2 = np.radians(df[lat_col])
    lon2 = np.radians(df[lon_col])

    # Diferenças angulares
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Fórmula de Haversine
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    distancias = raio_terra * c

    # Filtro
    mascara = distancias <= raio_km
    resultado = df[mascara].copy()

    if incluir_distancia:
        resultado["distancia_km"] = distancias[mascara]

    return resultado

In [ ]:
data_inicio = date.min.replace(df_crimes["ano_referencia"].min())
data_fim = date.max.replace(df_crimes["ano_referencia"].max())

df_crimes_tratado = df_crimes.copy()
df_crimes_tratado["DATA_OCORRENCIA_BO"] = pd.to_datetime(
    df_crimes_tratado["DATA_OCORRENCIA_BO"], errors="coerce"
)
df_crimes_tratado["LATITUDE"] = pd.to_numeric(
    df_crimes_tratado["LATITUDE"], errors="coerce"
)
df_crimes_tratado["LONGITUDE"] = pd.to_numeric(
    df_crimes_tratado["LONGITUDE"], errors="coerce"
)
df_crimes_tratado["BAIRRO"] = df_crimes_tratado["BAIRRO"].str.upper().str.strip()

mascara_universo = (
    df_crimes_tratado["DATA_OCORRENCIA_BO"].dt.date.between(data_inicio, data_fim)
    & df_crimes_tratado["LATITUDE"].notna()
    & df_crimes_tratado["LONGITUDE"].notna()
)

df_crimes_filtrado = (
    df_crimes_tratado[mascara_universo]
    .drop_duplicates(subset=["NUM_BO"], keep="first")
    .reset_index(drop=True)
)

df_crimes_filtrado = filtrar_por_raio(
    df_crimes_filtrado,
    lat_col="LATITUDE",
    lon_col="LONGITUDE",
    centro_lat=COORDENADAS_ORIGEM[0],
    centro_lon=COORDENADAS_ORIGEM[1],
    raio_km=RAIO_ANALISE_KM,
    incluir_distancia=True,
)

df_crimes_filtrado.head()

## Engenharia de variáveis

In [ ]:
def mapear_periodo(row: pd.Series):
    dt = row["DATA_HORA_OCORRENCIA_BO"]
    if pd.notna(dt):
        h = dt.hour
        if 6 <= h < 12:
            return "MANHÃ"
        if 12 <= h < 18:
            return "TARDE"
        if 18 <= h < 24:
            return "NOITE"
        return "MADRUGADA"
    # Fallback para a descrição original da SSP caso a hora esteja ausente
    desc = str(row["DESCR_PERIODO"]).strip().upper()
    if "MANH" in desc:
        return "MANHÃ"
    if "TARD" in desc:
        return "TARDE"
    if "NOIT" in desc:
        return "NOITE"
    if "MADR" in desc:
        return "MADRUGADA"
    return np.nan

In [ ]:
df_crimes_filtrado["DATA_HORA_OCORRENCIA_BO"] = pd.to_datetime(
    df_crimes_filtrado["DATA_OCORRENCIA_BO"].astype(str)
    + " "
    + df_crimes_filtrado["HORA_OCORRENCIA_BO"].astype(str),
    errors="coerce",
)

df_crimes_filtrado["MES_OCORRENCIA_BO"] = df_crimes_filtrado[
    "DATA_OCORRENCIA_BO"
].dt.month
df_crimes_filtrado["DIA_SEMANA_OCORRENCIA_BO"] = df_crimes_filtrado[
    "DATA_OCORRENCIA_BO"
].dt.day_of_week
df_crimes_filtrado["FLAG_FIM_DE_SEMANA_OCORRENCIA_BO"] = df_crimes_filtrado[
    "DIA_SEMANA_OCORRENCIA_BO"
].isin([5, 6])
df_crimes_filtrado["DESCR_PERIODO_OCORRENCIA_BO"] = df_crimes_filtrado.apply(
    mapear_periodo, axis=1
)

# Análise Exploratória de Dados (AED)

## Inspeção de estrutura

In [ ]:
# Sumário conciso do DataFrame

df_crimes_filtrado.info()

In [ ]:
# Estatísticas descritivas do DataFrame

df_crimes_filtrado.describe()

In [ ]:
# Saúde das colunas do DataFrame

s_nulos = df_crimes_filtrado.isna().sum()
s_porcentagens = ((s_nulos / len(df_crimes_filtrado)) * 100).round(2)

df_saude_colunas = pd.DataFrame({"Nulos": s_nulos, "Porcentagem (%)": s_porcentagens})

df_saude_colunas

In [ ]:
# Saúde da coluna NUM_BO (chave)

qtd_nulos = df_crimes_filtrado["NUM_BO"].isna().sum()
qtd_duplicados = df_crimes_filtrado["NUM_BO"].duplicated().sum()

print(f"Quantidade de registros com chave 'NUM_BO' nulos: {qtd_nulos} de {len(df_crimes_filtrado)} ({qtd_nulos/len(df_crimes_filtrado)*100:.2f}%)")
print(f"Quantidade de registros com chave 'NUM_BO' duplicados: {qtd_duplicados} de {len(df_crimes_filtrado)} ({qtd_duplicados/len(df_crimes_filtrado)*100:.2f}%)")

In [ ]:
# Saúde das colunas LATITUDE e LONGITUDE

qtd_valores_nulos_latitude = (df_crimes_filtrado["LATITUDE"].isna() | df_crimes_filtrado["LATITUDE"] == 0).sum()
qtd_valores_nulos_longitude = (df_crimes_filtrado["LONGITUDE"].isna() | df_crimes_filtrado["LONGITUDE"] == 0).sum()

print(f"Quantidade de valores nulos/inválidos na coluna 'LATITUDE': {qtd_valores_nulos_latitude} de {len(df_crimes_filtrado)} registros ({(qtd_valores_nulos_latitude / len(df_crimes_filtrado)) * 100:.2f}%)")
print(f"Quantidade de valores nulos/inválidos na coluna 'LONGITUDE': {qtd_valores_nulos_longitude} de {len(df_crimes_filtrado)} registros ({(qtd_valores_nulos_longitude / len(df_crimes_filtrado)) * 100:.2f}%)")

## Distribuição temporal e sazonalidade

### Histograma de data de ocorrência do BO

In [ ]:
sns.set_theme(style="whitegrid")

datas = df_crimes_filtrado["DATA_OCORRENCIA_BO"].dropna()

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    datas,
    bins=50,
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("Distribuição das ocorrências por data", fontsize=14, weight="bold")
ax.set_xlabel("Data da ocorrência", fontsize=11)
ax.set_ylabel("Quantidade de registros", fontsize=11)

ax.tick_params(axis="x", rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))

fig.tight_layout()
plt.show()

### Análise da hora de ocorrência do BO

In [ ]:
# Saúde da coluna HORA_OCORRENCIA_BO

qtd_horas_ocorrencia_nulas = df_crimes_filtrado["HORA_OCORRENCIA_BO"].isna().sum()

print(f"Quantidade de valores nulos na coluna 'HORA_OCORRENCIA_BO': {qtd_horas_ocorrencia_nulas} de {len(df_crimes_filtrado)} registros ({(qtd_horas_ocorrencia_nulas / len(df_crimes_filtrado)) * 100:.2f}%)")

In [ ]:
sns.set_theme(style="whitegrid")

horas = pd.to_datetime(
    df_crimes_filtrado["HORA_OCORRENCIA_BO"].astype(str),
    format="%H:%M:%S",
    errors="coerce",
).dropna()

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    horas,
    bins=50,
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("Distribuição das ocorrências por hora", fontsize=14, weight="bold")
ax.set_xlabel("Hora da ocorrência", fontsize=11)
ax.set_ylabel("Quantidade de registros", fontsize=11)

ax.tick_params(axis="x", rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

fig.tight_layout()
plt.show()

### Checando saúde da coluna de descrição do período

In [ ]:
# Saúde da coluna DESCR_PERIODO

qtd_descricoes_nulas = df_crimes_filtrado["DESCR_PERIODO"].isna()

print(f"Quantidade de valores nulos na coluna 'DESCR_PERIODO': {qtd_descricoes_nulas.sum()} de {len(df_crimes_filtrado)} registros ({(qtd_descricoes_nulas.sum() / len(df_crimes_filtrado)) * 100:.2f}%)")

In [ ]:
categorias_descr_periodo = df_crimes_filtrado["DESCR_PERIODO"].value_counts().head().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=categorias_descr_periodo.values,
    y=categorias_descr_periodo.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Descrições de período mais frequentes", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Descrição do período")

fig.tight_layout()
plt.show()

### Checando variáveis criadas

In [ ]:
# MES_OCORRENCIA_BO

sns.set_theme(style="whitegrid")

meses = df_crimes_filtrado["MES_OCORRENCIA_BO"].dropna().value_counts().sort_index()

meses.index = meses.index.map(
    {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro",
    }
)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=meses.index,
    y=meses.values,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="y", style="plain", useOffset=False, useLocale=True)

ax.set_title("Distribuição das ocorrências por mês", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Mês da ocorrência")

fig.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")

dias_da_semana = (
    df_crimes_filtrado["DIA_SEMANA_OCORRENCIA_BO"]
    .astype("Int64")
    .dropna()
    .value_counts()
    .sort_index()
)

dias_da_semana.index = dias_da_semana.index.map(
    {
        0: "Segunda-feira",
        1: "Terça-feira",
        2: "Quarta-feira",
        3: "Quinta-feira",
        4: "Sexta-feira",
        5: "Sábado",
        6: "Domingo",
    }
)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=dias_da_semana.index,
    y=dias_da_semana.values,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="y", style="plain", useOffset=False, useLocale=True)

ax.set_title(
    "Distribuição das ocorrências por dia da semana", fontsize=14, weight="bold"
)
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Dia da semana")

fig.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")

fim_semana = (
    df_crimes_filtrado["FLAG_FIM_DE_SEMANA_OCORRENCIA_BO"]
    .astype(str)
    .value_counts()
    .reindex(["False", "True"], fill_value=0)
)

fim_semana.index = ["Dias úteis", "Fim de semana"]

fig, ax = plt.subplots(figsize=(8, 6))

sns.barplot(
    x=fim_semana.index,
    y=fim_semana.values,
    color="#4C72B0",
    ax=ax,
)

ax.set_title("Distribuição das ocorrências por período do fim de semana", fontsize=14, weight="bold")
ax.set_xlabel("Período")
ax.set_ylabel("Quantidade de ocorrências")
ax.ticklabel_format(axis="y", style="plain", useOffset=False, useLocale=True)

fig.tight_layout()
plt.show()

In [ ]:
categorias_descr_periodo = df_crimes_filtrado["DESCR_PERIODO_OCORRENCIA_BO"].value_counts().head().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=categorias_descr_periodo.values,
    y=categorias_descr_periodo.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Descrições de período mais frequentes", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Descrição do período")

fig.tight_layout()
plt.show()

## Mapeamento de frequências

### Bairros

In [ ]:
# Saúde da coluna BAIRRO

qtd_bairros_nulos = df_crimes_filtrado["BAIRRO"].isna().sum()

print(f"Quantidade de valores nulos na coluna 'BAIRRO': {qtd_bairros_nulos} de {len(df_crimes_filtrado)} registros ({(qtd_bairros_nulos / len(df_crimes_filtrado)) * 100:.2f}%)")

In [ ]:
top_bairros = df_crimes_filtrado["BAIRRO"].value_counts().head(20).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=top_bairros.values,
    y=top_bairros.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Top 20 bairros com mais ocorrências", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Bairro")

fig.tight_layout()
plt.show()

### Rubricas

In [ ]:
top_rubricas = df_crimes_filtrado["RUBRICA"].value_counts().head(10).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=top_rubricas.values,
    y=top_rubricas.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Top 10 rubricas criminais mais frequentes", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Rubrica")

fig.tight_layout()
plt.show()

## Análises bivariadas

### Tabela de contingência Período x Rúbrica

In [ ]:
pd.crosstab(
    index=df_crimes_filtrado["DESCR_PERIODO"],
    columns=df_crimes_filtrado["RUBRICA"],
    margins=True,
    margins_name="Total",
)

## Especial: mapa de calor de ocorrências

In [ ]:
def calcular_zoom_start(raio_km: float) -> int:
    """Calcula o nível de zoom inicial para um mapa folium com base no raio em km."""
    if raio_km <= 0:
        raise ValueError("O raio deve ser maior que zero.")

    # Fórmula aproximada para calcular o nível de zoom
    zoom_start = int(13 - math.log10(raio_km / 2))

    # Garantir que o nível de zoom esteja dentro do intervalo permitido (0 a 18)
    return max(0, min(zoom_start, 18))

In [ ]:
mapa_calor = folium.Map(
    location=COORDENADAS_ORIGEM,
    zoom_start=calcular_zoom_start(RAIO_ANALISE_KM),
)

coordenadas = df_crimes_filtrado[["LATITUDE", "LONGITUDE"]].dropna()

for col in coordenadas.columns:
    coordenadas[col] = pd.to_numeric(coordenadas[col], errors="coerce")

coordenadas = coordenadas.dropna().values.tolist()

HeatMap(coordenadas, radius=10, blur=10).add_to(mapa_calor)

display(mapa_calor)

# Processamento e exportação de dados

In [ ]:
# Grid completo de datas e os 4 períodos
todas_datas = pd.date_range(
    start=df_crimes_filtrado['DATA_OCORRENCIA_BO'].min(),
    end=df_crimes_filtrado['DATA_OCORRENCIA_BO'].max(),
    freq='D'
).date
periodos = ['MADRUGADA', 'MANHÃ', 'TARDE', 'NOITE']
grid_completo = pd.MultiIndex.from_product(
    [todas_datas, periodos], 
    names=['DATA', 'DESCR_PERIODO_OCORRENCIA_BO']
).to_frame().reset_index(drop=True)

# Contagem agregada das ocorrências existentes
contagem_real = (
    df_crimes_filtrado.dropna(subset=['DESCR_PERIODO_OCORRENCIA_BO'])
    .groupby([df_crimes_filtrado['DATA_OCORRENCIA_BO'].dt.date, 'DESCR_PERIODO_OCORRENCIA_BO'])['NUM_BO']
    .count()
    .reset_index()
    .rename(columns={'DATA_OCORRENCIA_BO': 'DATA', 'NUM_BO': 'QTD_CRIMES'})
)

# Consolidação com preenchimento de zeros
df_agrupado_final = pd.merge(grid_completo, contagem_real, on=['DATA', 'DESCR_PERIODO_OCORRENCIA_BO'], how='left')
df_agrupado_final['QTD_CRIMES'] = df_agrupado_final['QTD_CRIMES'].fillna(0).astype(int)

# Definição do target binário
mediana = df_agrupado_final['QTD_CRIMES'].median()
df_agrupado_final['RISCO_ALTO'] = (df_agrupado_final['QTD_CRIMES'] > mediana).astype(int)

df_agrupado_final.head()

In [ ]:
arquivo_destino = pathlib.Path(DIR_DADOS_PROCESSADOS) / NOME_ARQUIVO_DATASET_PROCESSADO

df_agrupado_final.to_csv(arquivo_destino, sep=";", encoding="utf-8")